# Module 12: Agent Protocols - 01: Why agent-to-agent protocols exist

> **MLCourse - Agentic AI - Agent Patterns**

Every multi-agent system you have built in this course so far shares one
property: **you wrote both sides.** In
[`02_langgraph/06_multi_agent_systems`](../../02_langgraph/06_multi_agent_systems/README.md)
the supervisor and the workers are nodes in the *same graph*, sharing the same
process and the same Python objects. In `04_crewai`, every agent in a `Crew`
is instantiated by the same script.

That works because you control every agent in the system. **Agent-to-agent
protocols exist for the case where you don't.**

### The problem this module solves

Imagine your agent needs to book a flight. The airline's agent was built by
the airline, in a framework you have never heard of, running on infrastructure
you cannot see. You cannot import its Python class. You cannot inspect its
prompt. All you can do is **send it a message and get one back**.

That is a fundamentally different integration problem from anything earlier in
this course, and it has a name: **agent interoperability**. This module covers
**A2A (Agent2Agent)**, the protocol Google introduced in 2025 specifically for
this case, now under Linux Foundation stewardship with contributions from
dozens of companies.

### What you will learn in this module

1. Why "just call the other agent's function" stops working across
   organisational boundaries.
2. A2A's core objects: **Agent Cards**, **Tasks**, and **capability
   discovery**.
3. Two local agents, implemented from scratch, exchanging real A2A-shaped
   messages - no hosted service required.
4. How A2A differs from MCP, and why the two are complements, not rivals.

### An important scoping note

We are not going to install the `a2a-sdk` or spin up a hosted A2A server. The
point of this module is to understand the **message shapes and the
interaction pattern**, which we build by hand in plain Python and JSON. That
is enough to see exactly what problem the protocol solves - and it means this
module needs no API key and no network access.

No LLM calls in this notebook. Notebook 03 optionally uses one small Groq call
to make an agent's *decision-making* look real; everything about the protocol
itself is deterministic Python.

### Setup


In [ ]:
import json
import uuid
from dataclasses import dataclass, field, asdict
from typing import Any, Optional

print("Module 12: Agent Protocols")
print("No API key needed for this notebook.")


### 1. Why "just call a function" breaks down

Every integration pattern earlier in this course assumes a shared runtime:

| Pattern | What's shared |
|---|---|
| LangGraph subgraph (`02_langgraph/07`) | the same compiled graph, the same state object |
| CrewAI crew (`04_crewai`) | the same Python process, the same `Crew` instance |
| MCP tool call (`04_crewai/.../04_mcp_integration`) | a protocol, but the *caller* still owns the interaction - the tool has no autonomy |

A2A is for the fourth case, which none of those cover: **two independently
built, independently hosted agents, owned by different parties, that need to
delegate real work to each other** - one agent asking another to actually
*do* something, potentially over minutes or hours, potentially asking
clarifying questions along the way.

Three properties fall out of that requirement, and A2A's design follows
directly from them:

1. **No shared code.** Neither agent can import the other's classes. The only
   contact surface is messages over HTTP, in a format both sides agree on in
   advance.
2. **No shared trust, by default.** The airline's agent should not have to
   trust your agent's account of what it can do - it needs a discoverable,
   independently-checkable description.
3. **Work can outlive a single request/response.** Booking a flight might
   involve "let me check availability... I need your preferred time... here
   is a confirmation" - a genuine multi-turn negotiation with state, not a
   single function call.

Those three needs map directly onto A2A's three core objects.

### 2. The three core objects

### Agent Card - capability discovery

Before you can delegate work to an unfamiliar agent, you need to know what it
can do, without reading its source code. An **Agent Card** is a small JSON
document, conventionally published at a well-known URL
(`/.well-known/agent-card.json`), that answers exactly that:

- who the agent is (name, description, provider)
- what it can do (a list of **skills**, each with an id, description, and
  example prompts)
- how to reach it (its endpoint URL)
- what input/output formats and auth schemes it supports

This is the A2A analogue of an MCP server's `list_tools()` response - except
it describes an *agent's* capabilities, discoverable **before any connection
is made**, rather than a live server's tool list fetched *after* connecting.

### Task - the unit of delegated work

A **Task** is the object that represents one piece of delegated work, from
creation to completion. It has:

- a unique `id`
- a `status` (`submitted`, `working`, `input-required`, `completed`,
  `failed`, ...)
- a history of **messages** exchanged about it
- optional **artifacts** - the actual output the task produced

The status machine is the part that matters most. A Task is not "send a
prompt, get a completion" - it can sit in `input-required` for as long as it
takes a human or another system to answer a clarifying question, which is
precisely the multi-turn negotiation property from section 1.

### Message - one turn in the conversation about a Task

A **Message** carries one turn: a `role` (`user` or `agent`), a list of
**Parts** (text, structured data, or a file), and the `taskId` it belongs to.
Parts, not a single string, because a real exchange often needs to carry both
a natural-language explanation and a structured payload in the same turn.

### Modelling the three A2A objects in plain Python


In [ ]:
# These dataclasses mirror the real A2A JSON shapes closely enough to make
# the concepts concrete, without depending on the a2a-sdk package.

@dataclass
class Skill:
    id: str
    name: str
    description: str
    examples: list[str] = field(default_factory=list)


@dataclass
class AgentCard:
    """Published, discoverable BEFORE any connection is made."""
    name: str
    description: str
    url: str
    version: str
    skills: list[Skill] = field(default_factory=list)
    default_input_modes: list[str] = field(default_factory=lambda: ["text/plain"])
    default_output_modes: list[str] = field(default_factory=lambda: ["text/plain"])


@dataclass
class Part:
    kind: str            # "text" | "data"
    text: Optional[str] = None
    data: Optional[dict] = None


@dataclass
class Message:
    role: str             # "user" | "agent"
    parts: list[Part]
    task_id: str
    message_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])


@dataclass
class Task:
    id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    status: str = "submitted"
    history: list[Message] = field(default_factory=list)
    artifacts: list[dict] = field(default_factory=list)


# A worked example: a "flight booking" agent's card.
flight_agent_card = AgentCard(
    name="SkyBooker",
    description="Books and manages flight reservations.",
    url="https://skybooker.example/a2a",
    version="1.0.0",
    skills=[
        Skill(id="search-flights", name="Search Flights",
             description="Find flights between two cities on a given date.",
             examples=["Find flights from JFK to LHR on Dec 3"]),
        Skill(id="book-flight", name="Book Flight",
             description="Reserve a specific flight for a named passenger.",
             examples=["Book flight SK123 for Ada Lovelace"]),
    ],
)

print("Published Agent Card (this is what discovery returns):\n")
print(json.dumps(asdict(flight_agent_card), indent=2))


### Reading the card

Notice what is and is not in there. A **capability** (`search-flights`,
`book-flight`) is described declaratively - a name, a description, example
prompts - not as a function signature. That is deliberate: the calling agent
does not need to know *how* SkyBooker searches flights, only *that* it can,
and roughly what request shapes it understands. This is looser coupling than
an MCP tool's typed input schema, and section 4 explains exactly why that
looseness is the point.

### 3. The interaction pattern, end to end

Putting the three objects together, an A2A exchange follows this shape:

```
1. DISCOVER   caller fetches the other agent's Agent Card
2. DECIDE     caller checks: does a skill in this card match what I need?
3. DELEGATE   caller creates a Task, sends the first Message
4. NEGOTIATE  (optional) the agent may reply with status=input-required
              and a clarifying Message; this can repeat several turns
5. COMPLETE   the agent replies with status=completed and an artifact,
              or status=failed
```

Compare this to a plain function call: a function call is step 3 and 5 only,
with no discovery step and no room for a mid-call clarifying question. Steps 1
and 4 are the two things a same-process integration never needed and a
cross-organisation one cannot do without.

### Simulating steps 1-3: discover, decide, delegate


In [ ]:
def discover(card: AgentCard, need: str) -> Optional[Skill]:
    """Step 1+2: does this agent's published card cover what we need?
    A real implementation would do semantic matching (see module 05,
    Semantic Routing) against skill descriptions; a simple substring
    check is enough to demonstrate the mechanism."""
    need_lower = need.lower()
    for skill in card.skills:
        haystack = f"{skill.name} {skill.description}".lower()
        if any(w in haystack for w in need_lower.split()):
            return skill
    return None


need = "I need to book a flight for a passenger"
matched_skill = discover(flight_agent_card, need)

print(f"Caller's need : {need!r}")
print(f"Matched skill : {matched_skill.id if matched_skill else None}")
print(f"  -> {matched_skill.description if matched_skill else 'no match'}")

# Step 3: DELEGATE. Create a Task and send the first message.
task = Task()
first_message = Message(
    role="user",
    parts=[Part(kind="text", text="Book flight SK123 for Ada Lovelace")],
    task_id=task.id,
)
task.history.append(first_message)
task.status = "submitted"

print(f"\nTask created: id={task.id}, status={task.status}")
print(f"First message: {first_message.parts[0].text!r}")


### Key takeaways

- Every prior multi-agent pattern in this course shared a runtime. A2A is for
  when the two agents **don't** - different codebases, different
  organisations, connected only by messages.
- **Agent Card** - discoverable capability description, checked *before* any
  connection. The A2A analogue of `list_tools()`, but describing an agent, not
  a tool server.
- **Task** - the unit of delegated work, with a status machine
  (`submitted → working → input-required? → completed/failed`) that allows
  work to genuinely span multiple turns.
- **Message** - one turn, carrying one or more typed **Parts**, always tied to
  a `taskId`.
- The interaction is discover → decide → delegate → (negotiate) → complete -
  strictly more than a function call's request/response.

**Next:** `02_two_local_agents.ipynb` - two complete local agents (a
"traveler" and "SkyBooker") built from scratch, exchanging real A2A-shaped
messages including a genuine multi-turn negotiation, with no hosted service
involved.